[![Ouvrir dans Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DEDJIABDOUL/omicaa-lab-machine-learning-bioinformatics/blob/main/notebooks/02_numpy_pandas_exploration.ipynb)

# Notebook 02 : NumPy, pandas et exploration d'un jeu de données biologique

**Omicaa Lab | Machine Learning et Deep Learning pour les biologistes** | Auteur : DEDJI ABDOUL BATINOU

## Introduction scientifique

Une mesure biologique devient exploitable par un algorithme lorsqu'elle est représentée par un tableau de nombres dont chaque ligne est une observation et chaque colonne une caractéristique. NumPy fournit le tableau numérique efficace, pandas ajoute les noms, les types et les outils de manipulation. Ce notebook ouvre le projet fil rouge du parcours : le jeu de données Breast Cancer Wisconsin (Diagnostic), dans lequel des mesures morphologiques de noyaux cellulaires, obtenues par analyse d'images de ponction à l'aiguille fine, sont associées à un diagnostic.

## Objectifs techniques

1. Créer et manipuler des tableaux NumPy : forme, type, indexation, masques, opérations vectorisées, broadcasting.
2. Construire et interroger un DataFrame pandas : colonnes, index, types, filtrage, jointure, regroupement.
3. Distinguer observation, caractéristique, étiquette et mesure biologique.
4. Réaliser le contrôle qualité initial : dimensions, types, doublons, valeurs manquantes, distribution de la cible.

## Prérequis

* Notebook 01 terminé, ou connaissance des variables, listes, fonctions.

## Résultat attendu

Un diagnostic de structure du jeu de données, validé par des assertions, et une première décision justifiée sur le codage de la variable cible.

## 1. Contrôle de l'environnement

In [1]:
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python      :", platform.python_version())
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

Python      : 3.12.3
NumPy       : 2.4.4
pandas      : 3.0.2
Matplotlib  : 3.10.8
scikit-learn: 1.8.0


## 2. NumPy : le tableau numérique

Un `ndarray` est un tableau de valeurs de **même type**, stockées de façon contiguë. Il possède une **forme** (`shape`), un nombre de dimensions (`ndim`) et un type (`dtype`). Une matrice d'expression de 4 échantillons et 3 gènes a la forme `(4, 3)` : les lignes sont les échantillons, les colonnes sont les gènes. Cette convention, `(n, p)`, sera celle de toute la formation.

Les valeurs ci-dessous sont **fictives** et servent uniquement à illustrer les opérations.

In [2]:
expr = np.array([[5.0, 7.5, 2.1],
                 [6.2, 7.9, 1.8],
                 [4.8, 9.1, 2.5],
                 [5.5, 8.4, 2.0]])
print("forme :", expr.shape, "| dimensions :", expr.ndim, "| type :", expr.dtype, "| éléments :", expr.size)

print("ligne 0         :", expr[0])
print("colonne 1       :", expr[:, 1])
print("sous-tableau    :", expr[1:3, 0:2].tolist())
print("dernier élément :", expr[-1, -1])

forme : (4, 3) | dimensions : 2 | type : float64 | éléments : 12
ligne 0         : [5.  7.5 2.1]
colonne 1       : [7.5 7.9 9.1 8.4]
sous-tableau    : [[6.2, 7.9], [4.8, 9.1]]
dernier élément : 2.0


**Opérations vectorisées.** Une opération appliquée à un tableau s'applique à tous ses éléments, sans boucle explicite. C'est plus court, plus lisible et beaucoup plus rapide. L'argument `axis` choisit le sens du calcul : `axis=0` agrège **sur les lignes** (un résultat par colonne), `axis=1` agrège sur les colonnes (un résultat par ligne).

In [3]:
print("moyenne par gène (axis=0)      :", expr.mean(axis=0).round(3))
print("moyenne par échantillon (axis=1):", expr.mean(axis=1).round(3))
print("log2 de la matrice               :\n", np.log2(expr).round(2))

masque = expr[:, 2] < 2.2          # tableau booléen
print("échantillons avec gène 3 < 2.2 :", masque, "->", np.where(masque)[0])
print("sélection par masque :\n", expr[masque])

moyenne par gène (axis=0)      : [5.375 8.225 2.1  ]
moyenne par échantillon (axis=1): [4.867 5.3   5.467 5.3  ]
log2 de la matrice               :
 [[2.32 2.91 1.07]
 [2.63 2.98 0.85]
 [2.26 3.19 1.32]
 [2.46 3.07 1.  ]]
échantillons avec gène 3 < 2.2 : [ True  True False  True] -> [0 1 3]
sélection par masque :
 [[5.  7.5 2.1]
 [6.2 7.9 1.8]
 [5.5 8.4 2. ]]


**Broadcasting.** Lorsque deux tableaux n'ont pas la même forme, NumPy étend automatiquement le plus petit si les dimensions sont compatibles. Soustraire la moyenne de chaque colonne (forme `(3,)`) à la matrice (forme `(4, 3)`) centre chaque gène. Cette opération est exactement celle que réalisera `StandardScaler` plus tard.

In [4]:
moyennes = expr.mean(axis=0)
centre = expr - moyennes
print("forme moyennes :", moyennes.shape, "| forme résultat :", centre.shape)
print("moyennes après centrage :", centre.mean(axis=0).round(12))
assert np.allclose(centre.mean(axis=0), 0)

forme moyennes : (3,) | forme résultat : (4, 3)
moyennes après centrage : [ 0.  0. -0.]


## 3. pandas : le tableau étiqueté

Un `DataFrame` associe à un tableau des **noms de colonnes** et un **index** de lignes, et autorise des colonnes de types différents. Les mêmes données sont plus faciles à lire et moins sujettes aux erreurs de position.

In [5]:
df_demo = pd.DataFrame(expr, columns=["BRCA1", "TP53", "MYC"], index=["E1", "E2", "E3", "E4"])
df_demo["groupe"] = ["tumeur", "tumeur", "normal", "normal"]
print(df_demo)
print()
print(df_demo.dtypes)
print()
print("filtrage :", df_demo[df_demo["TP53"] > 8].index.tolist())
print(df_demo.groupby("groupe")[["BRCA1", "TP53", "MYC"]].mean().round(2))

    BRCA1  TP53  MYC  groupe
E1    5.0   7.5  2.1  tumeur
E2    6.2   7.9  1.8  tumeur
E3    4.8   9.1  2.5  normal
E4    5.5   8.4  2.0  normal

BRCA1     float64
TP53      float64
MYC       float64
groupe        str
dtype: object

filtrage : ['E3', 'E4']
        BRCA1  TP53   MYC
groupe                   
normal   5.15  8.75  2.25
tumeur   5.60  7.70  1.95


## 4. Le jeu de données fil rouge : Breast Cancer Wisconsin (Diagnostic)

**Provenance.** Données créées par W. H. Wolberg, W. N. Street et O. L. Mangasarian, déposées dans le UCI Machine Learning Repository (identifiant 17), publiées sous licence **CC BY 4.0**. scikit-learn en distribue une copie intégrée, ce qui garantit un accès hors ligne et reproductible.

**Contenu.** 569 observations, 30 caractéristiques numériques et un diagnostic. Dix propriétés morphologiques du noyau cellulaire (rayon, texture, périmètre, aire, régularité, compacité, concavité, points concaves, symétrie, dimension fractale) sont résumées par trois statistiques : moyenne (`mean`), erreur standard (`error`) et valeur la plus extrême (`worst`), soit 3 x 10 = 30 colonnes.

**Pourquoi ce jeu de données.** Il est tabulaire, de taille modérée, sans valeur manquante et sans variable catégorielle : il permet d'apprendre le pipeline complet sur CPU sans masquer la méthode derrière des difficultés de nettoyage. Les valeurs manquantes, les variables catégorielles et la forte dimensionnalité seront traitées dans les notebooks 03 et 07.

**Limites à garder en tête.** Cette copie ne contient pas d'identifiant patient. On ne peut donc pas vérifier qu'une même patiente n'apparaît pas plusieurs fois, ce qui pourrait rendre un partitionnement aléatoire trop optimiste. Les résultats ne constituent pas une validation clinique.

In [6]:
from sklearn.datasets import load_breast_cancer

bunch = load_breast_cancer(as_frame=True)
X = bunch.data.copy()
print("noms de classes scikit-learn :", list(bunch.target_names))
print(bunch.target.value_counts().sort_index().to_dict(), "(0 = malignant, 1 = benign)")

noms de classes scikit-learn : [np.str_('malignant'), np.str_('benign')]
{0: 212, 1: 357} (0 = malignant, 1 = benign)


**Décision de codage de la cible.** Dans scikit-learn, `0` désigne *malignant* et `1` désigne *benign*. Or les métriques `precision`, `recall` et `f1` considèrent par défaut la classe `1` comme **classe positive**. En clinique, on souhaite mesurer la capacité à détecter la tumeur maligne : elle doit donc être la classe positive. On recode : `y = 1` pour *maligne*, `y = 0` pour *bénigne*. Ce choix sera conservé dans tous les notebooks.

In [7]:
y = (bunch.target == 0).astype(int).rename("maligne")
print(y.value_counts().sort_index().rename({0: "benigne (0)", 1: "maligne (1)"}))
assert set(y.unique()) == {0, 1}
assert len(X) == len(y) == 569

maligne
benigne (0)    357
maligne (1)    212
Name: count, dtype: int64


## 5. Observation, caractéristique, étiquette, mesure

| Terme | Définition | Ici |
|---|---|---|
| Observation | une ligne, une unité statistique | un échantillon cellulaire analysé |
| Caractéristique (feature) | une colonne d'entrée du modèle | `mean radius`, `worst texture`, ... |
| Étiquette (label, cible) | la valeur à prédire | `maligne` |
| Mesure biologique | la grandeur physique derrière la colonne | dimension d'un noyau, issue de l'analyse d'image |

Une caractéristique est une **représentation** numérique d'une mesure, et non la mesure elle-même. Les 30 colonnes sont calculées à partir des mêmes noyaux et sont donc fortement corrélées, ce qui aura des conséquences sur l'interprétation des modèles.

## 6. Contrôle qualité, étape 1 : structure, identifiants, doublons

In [8]:
print("forme (n, p)        :", X.shape)
print("types présents      :", X.dtypes.value_counts().to_dict())
print("lignes dupliquées   :", int(X.duplicated().sum()))
print("index unique        :", X.index.is_unique)
print("noms de colonnes uniques :", X.columns.is_unique)
print("cible dans X ?      :", "maligne" in X.columns or "target" in X.columns)

assert X.shape == (569, 30)
assert X.columns.is_unique
assert "maligne" not in X.columns and "target" not in X.columns
assert all(np.issubdtype(t, np.floating) for t in X.dtypes)
X.iloc[:3, :6]

forme (n, p)        : (569, 30)
types présents      : {dtype('float64'): 30}
lignes dupliquées   : 0
index unique        : True
noms de colonnes uniques : True
cible dans X ?      : False


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness
0,17.99,10.38,122.8,1001.0,0.11840,0.27760
1,20.57,17.77,132.9,1326.0,0.08474,0.07864
2,19.69,21.25,130.0,1203.0,0.10960,0.15990


## 7. Contrôle qualité, étape 2 : valeurs manquantes

On compte les valeurs manquantes par colonne. Elles ne sont pas toujours codées `NaN` : un `0` ou un `-999` peuvent jouer ce rôle, ce qui justifie de vérifier aussi les valeurs impossibles.

In [9]:
print("valeurs manquantes (total) :", int(X.isna().sum().sum()))
print("minimum par colonne, 5 plus petits :")
print(X.min().sort_values().head(5).round(5))
print("colonnes contenant des zéros :", X.columns[(X == 0).any()].tolist())

valeurs manquantes (total) : 0
minimum par colonne, 5 plus petits :
mean concavity          0.0
mean concave points     0.0
concave points error    0.0
concavity error         0.0
worst concave points    0.0
dtype: float64
colonnes contenant des zéros : ['mean concavity', 'mean concave points', 'concavity error', 'concave points error', 'worst concavity', 'worst concave points']


Les colonnes de concavité et de points concaves contiennent des zéros. Biologiquement, c'est plausible : un noyau parfaitement convexe n'a aucune concavité. Ces zéros sont donc des **mesures valides** et non des valeurs manquantes déguisées. Cette distinction illustre la différence entre anomalie technique et observation biologiquement plausible, développée au notebook 03.

**Simulation pédagogique.** Pour apprendre à manipuler des valeurs manquantes sans modifier les données réelles, on en introduit artificiellement dans une **copie**. Ce qui suit est une simulation, il ne s'agit pas de données observées.

In [10]:
rng = np.random.default_rng(SEED)
X_trous = X.copy()
masque_nan = rng.random(X_trous.shape) < 0.02
X_trous = X_trous.mask(masque_nan)

print("valeurs manquantes simulées :", int(X_trous.isna().sum().sum()), "sur", X_trous.size)
print("colonnes les plus touchées :")
print(X_trous.isna().sum().sort_values(ascending=False).head(3))
print("lignes complètes si dropna :", len(X_trous.dropna()), "sur", len(X_trous))

valeurs manquantes simulées : 329 sur 17070
colonnes les plus touchées :
worst texture      17
concavity error    17
worst perimeter    17
dtype: int64
lignes complètes si dropna : 317 sur 569


Supprimer les lignes incomplètes (`dropna`) élimine ici environ 44 % des observations (317 lignes conservées sur 569), alors que seuls 2 % des cellules sont vides. Pour les jeux de données de petite taille, on **impute** plutôt (notebook 03), en estimant la valeur de remplacement sur les données d'entraînement uniquement.

## 8. Contrôle qualité, étape 3 : statistiques descriptives et cible

In [11]:
resume = X.describe().T[["mean", "std", "min", "50%", "max"]]
print(resume.round(3).head(8))
print()
print("ordres de grandeur : aire moyenne =", round(X["mean area"].mean(), 1),
      "| finesse moyenne =", round(X["mean smoothness"].mean(), 4))

                        mean      std      min      50%       max
mean radius           14.127    3.524    6.981   13.370    28.110
mean texture          19.290    4.301    9.710   18.840    39.280
mean perimeter        91.969   24.299   43.790   86.240   188.500
mean area            654.889  351.914  143.500  551.100  2501.000
mean smoothness        0.096    0.014    0.053    0.096     0.163
mean compactness       0.104    0.053    0.019    0.093     0.345
mean concavity         0.089    0.080    0.000    0.062     0.427
mean concave points    0.049    0.039    0.000    0.034     0.201

ordres de grandeur : aire moyenne = 654.9 | finesse moyenne = 0.0964


Les échelles diffèrent de plusieurs ordres de grandeur : une aire se compte en centaines, une régularité en centièmes. Pour les algorithmes fondés sur des distances ou des gradients (régression logistique, réseaux de neurones), cette hétérogénéité exige une **mise à l'échelle**. Les arbres de décision n'en ont pas besoin.

In [12]:
effectifs = y.value_counts().sort_index()
proportions = y.value_counts(normalize=True).sort_index()
tableau = pd.DataFrame({"effectif": effectifs, "proportion": proportions.round(3)})
tableau.index = ["benigne (0)", "maligne (1)"]   # renommer l'index après construction, sans réalignement
print(tableau)
assert tableau["effectif"].sum() == len(y)
taux_majoritaire = proportions.max()
print(f"taux de la classe majoritaire : {taux_majoritaire:.3f}")

             effectif  proportion
benigne (0)       357       0.627
maligne (1)       212       0.373
taux de la classe majoritaire : 0.627


Une règle qui prédirait toujours *bénigne* aurait une accuracy d'environ 0,627. Ce seuil est le **plancher** à dépasser : toute accuracy proche de cette valeur signale un modèle sans valeur. Les classes sont déséquilibrées de façon modérée, ce qui justifie l'usage de partitions **stratifiées**.

## 9. Jointure et regroupement : annoter les caractéristiques

Les 30 colonnes ont une structure : une propriété morphologique et une statistique. On construit une table d'annotation et on la joint aux résultats d'un calcul. La jointure (`merge`) aligne deux tables sur une clé commune.

In [13]:
annot = pd.DataFrame({"colonne": X.columns})
annot["statistique"] = annot["colonne"].str.extract(r"^(mean|worst)", expand=False).fillna("error")
annot["propriete"] = (annot["colonne"].str.replace(r"^(mean|worst) ", "", regex=True)
                                       .str.replace(r"^ ?(.*) error$", r"\1", regex=True))
print(annot.head(4).to_string(index=False))
print(annot["statistique"].value_counts().to_dict())

diff = (X[y == 1].mean() - X[y == 0].mean()) / X.std()    # écart standardisé maligne moins bénigne
table_diff = pd.DataFrame({"colonne": diff.index, "ecart_standardise": diff.values})
fusion = table_diff.merge(annot, on="colonne", how="left", validate="one_to_one")
assert fusion["statistique"].notna().all()
print(fusion.groupby("statistique")["ecart_standardise"].agg(["mean", "max"]).round(2))
fusion.sort_values("ecart_standardise", ascending=False).head(5)[["colonne", "ecart_standardise"]].round(2)

       colonne statistique propriete
   mean radius        mean    radius
  mean texture        mean   texture
mean perimeter        mean perimeter
     mean area        mean      area


{'mean': 10, 'error': 10, 'worst': 10}


             mean   max
statistique            
error        0.54  1.17
mean         1.10  1.60
worst        1.23  1.64


,colonne,ecart_standardise
27,worst concave points,1.64
22,worst perimeter,1.62
7,mean concave points,1.60
20,worst radius,1.60
2,mean perimeter,1.53


Les colonnes `worst` et `mean` séparent davantage les deux groupes que les colonnes `error`. Ce constat est **descriptif** : il décrit une association statistique dans cet échantillon et n'établit aucun mécanisme biologique.

## 10. Exercices progressifs

**Exercice 1.** Créez un tableau NumPy de forme `(3, 4)` contenant les entiers de 1 à 12, puis calculez la somme de chaque colonne.

*Solution de l'exercice 1*

In [14]:
a = np.arange(1, 13).reshape(3, 4)
somme_colonnes = a.sum(axis=0)
print(a, somme_colonnes)
assert somme_colonnes.tolist() == [15, 18, 21, 24]

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]] [15 18 21 24]


**Exercice 2.** Combien d'observations ont une `mean radius` supérieure à 15, et quelle proportion de ces observations est maligne ?

*Solution de l'exercice 2*

In [15]:
grands = X["mean radius"] > 15
print("effectif :", int(grands.sum()), "| proportion maligne :", round(float(y[grands].mean()), 3))
assert grands.sum() == int((X["mean radius"] > 15).sum())

effectif : 173 | proportion maligne : 0.931


**Exercice 3.** Calculez, pour chaque classe, la médiane de `mean area`. Que constatez-vous ?

*Solution de l'exercice 3*

In [16]:
med = X.groupby(y.map({0: "benigne", 1: "maligne"}))["mean area"].median()
print(med.round(1))
assert med["maligne"] > med["benigne"]

maligne
benigne    458.4
maligne    932.0
Name: mean area, dtype: float64


**Exercice 4.** Vérifiez par un `assert` que les index de `X` et de `y` sont identiques, puis expliquez pourquoi cette vérification protège contre une erreur d'alignement.

*Solution de l'exercice 4*

In [17]:
assert X.index.equals(y.index)
print("index alignés. Un décalage associerait la mauvaise étiquette à une observation sans lever d'erreur.")

index alignés. Un décalage associerait la mauvaise étiquette à une observation sans lever d'erreur.


## 11. Synthèse des compétences acquises

* Lire la forme, le type et la dimension d'un tableau ; indexer, filtrer et agréger avec `axis`.
* Comprendre le broadcasting comme principe du centrage des variables.
* Distinguer observation, caractéristique, étiquette et mesure biologique.
* Mener un contrôle qualité initial et vérifier ses conclusions par des `assert`.
* Recoder une cible pour que la classe positive corresponde à la question scientifique.

**Suite.** Le notebook 03 visualise ces données, distingue anomalie et valeur atypique, puis construit les premiers prétraitements.